# PyTorch Neural-Network Systematics Example

This notebook implements the one-dimensional nuisance-ratio model of Sect. 3.2 of *Learning new physics from an imperfect machine* using PyTorch. It follows the parametric-classifier strategy in the linked `Learning_NP` repository:

$$
\log \widehat r(x;\nu)
=\nu\,\widehat\delta_1(x)
+\frac{\nu^2}{2}\widehat\delta_2(x).
$$

A separate linear-only model is trained, so its $\widehat\delta_1$ cannot be deformed by the quadratic fit. The normalization nuisance is fixed to zero in this example.

In [1]:
import copy
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, TensorDataset

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})

/data/marcol/anaconda3/envs/flk_torch113_cu116/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

The architecture, nuisance points, and sample sizes follow Sect. 3.2. The optimization settings below are exposed so that training time and early stopping can be adjusted without changing the model.

In [2]:
seed = 123
rng_linear = np.random.default_rng(seed)
rng_quadratic = np.random.default_rng(seed + 1)
torch.manual_seed(seed)

n_central = 20_000
n_varied = 20_000
linear_nu_values = np.array([-0.10, -0.05, 0.05, 0.10])
quadratic_nu_values = np.array([-0.30, -0.05, 0.05, 0.30])

hidden_units = 4
batch_size = 16_384
learning_rate = 1e-2
max_epochs = 1000
early_stopping_patience = 150
validation_fraction = 0.2
weight_clip = 100.0
loss_scale = 100.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_max = 6.0
x_grid = np.linspace(0.0, x_max, 250).reshape(-1, 1)

print("device:", device)
print("linear nuisance points:", linear_nu_values)
print("quadratic nuisance points:", quadratic_nu_values)

device: cuda
linear nuisance points: [-0.1  -0.05  0.05  0.1 ]
quadratic nuisance points: [-0.3  -0.05  0.05  0.3 ]


## 2. Define the Toy Model

The central distribution is exponential with unit scale. The scale nuisance changes its scale to $e^\nu$, giving

$$
\log r(x;\nu)=x(1-e^{-\nu})-\nu,\qquad
\delta_1(x)=x-1,\qquad\delta_2(x)=-x.
$$

In [3]:
def sample_exponential(n_events, nu, random_generator):
    """Draw events from the nuisance-varied exponential distribution."""
    scale = np.exp(float(nu))
    return random_generator.exponential(scale=scale, size=(n_events, 1))


def exact_log_ratio(x, nu):
    """Evaluate the exact log p(x|nu) / p(x|0)."""
    x = np.asarray(x).reshape(-1)
    nu = float(nu)
    return x * (1.0 - np.exp(-nu)) - nu


def exact_delta1(x):
    """Evaluate the exact first Taylor coefficient."""
    return np.asarray(x).reshape(-1) - 1.0


def exact_delta2(x):
    """Evaluate the exact second Taylor coefficient."""
    return -np.asarray(x).reshape(-1)

## 3. Build the Nuisance-Tagged Samples

For every nuisance value, generate 20,000 central events and 20,000 varied events. Central events have label 0 and varied events have label 1. The nuisance value is stored beside every event but is used only in the polynomial score, not as an input to the coefficient networks.

In [4]:
def build_classification_sample(nuisance_values, random_generator):
    """Stack central and varied samples for all nuisance points."""
    features = []
    nuisances = []
    labels = []

    for nu in nuisance_values:
        x_central = sample_exponential(n_central, 0.0, random_generator)
        x_varied = sample_exponential(n_varied, nu, random_generator)
        features.extend([x_central, x_varied])
        nuisances.extend([
            np.full((n_central, 1), float(nu)),
            np.full((n_varied, 1), float(nu)),
        ])
        labels.extend([
            np.zeros((n_central, 1)),
            np.ones((n_varied, 1)),
        ])

    return (
        np.ascontiguousarray(np.vstack(features), dtype=np.float32),
        np.ascontiguousarray(np.vstack(nuisances), dtype=np.float32),
        np.ascontiguousarray(np.vstack(labels), dtype=np.float32),
    )


linear_sample = build_classification_sample(
    linear_nu_values, rng_linear
)
quadratic_sample = build_classification_sample(
    quadratic_nu_values, rng_quadratic
)
print("linear sample shape:", linear_sample[0].shape)
print("quadratic sample shape:", quadratic_sample[0].shape)

linear sample shape: (160000, 1)
quadratic sample shape: (160000, 1)


## 4. Define the Coefficient Networks

Each coefficient is represented by its own `(1, 4, 1)` fully connected ReLU network. The linear model contains only $\widehat\delta_1^L$. The quadratic model contains two distinct networks $\widehat\delta_1^Q$ and $\widehat\delta_2^Q$, optimized jointly through their combined score.

In [5]:
class CoefficientNetwork(nn.Module):
    """One (1, 4, 1) coefficient network."""

    def __init__(self, width):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(1, width),
            nn.ReLU(),
            nn.Linear(width, 1),
        )

    def forward(self, x):
        return self.layers(x)


class TaylorRatioNetwork(nn.Module):
    """Neural Taylor model for the nuisance log-ratio."""

    def __init__(self, order, width):
        super().__init__()
        if order not in (1, 2):
            raise ValueError("order must be 1 or 2")
        self.order = order
        self.delta1 = CoefficientNetwork(width)
        self.delta2 = CoefficientNetwork(width) if order == 2 else None

    def coefficients(self, x):
        delta1 = self.delta1(x)
        delta2 = self.delta2(x) if self.delta2 is not None else None
        return delta1, delta2

    def forward(self, x, nu):
        delta1, delta2 = self.coefficients(x)
        log_ratio = nu * delta1
        if delta2 is not None:
            log_ratio = log_ratio + 0.5 * nu**2 * delta2
        return log_ratio

## 5. Implement Eq. 17

The continuous classifier is

$$
c(x;\nu)=\frac{1}{1+\widehat r(x;\nu)}
=\operatorname{sigmoid}[-\log\widehat r(x;\nu)].
$$

With label 0 for central events and label 1 for varied events, the equally weighted empirical loss is

$$
L=\sum_{\rm varied}c^2+\sum_{\rm central}(1-c)^2.
$$

The factor `loss_scale` matches the convention used in the reference implementation and does not change the minimizer.

In [6]:
def equation17_loss(log_ratio, labels):
    """Evaluate the squared classifier loss of Eq. 17."""
    classifier = torch.sigmoid(-log_ratio)
    event_loss = (
        labels * classifier**2
        + (1.0 - labels) * (1.0 - classifier)**2
    )
    return loss_scale * torch.mean(event_loss)

## 6. Train with Validation and Early Stopping

The validation loss uses held-out simulated events and the same classifier objective. It does not require access to the analytic density ratio. The best validation checkpoint is restored after training.

In [7]:
def make_data_loaders(sample, split_seed):
    """Create deterministic training and validation loaders."""
    tensors = [torch.from_numpy(array) for array in sample]
    dataset = TensorDataset(*tensors)
    generator = torch.Generator().manual_seed(split_seed)
    indices = torch.randperm(len(dataset), generator=generator).tolist()
    n_validation = int(validation_fraction * len(dataset))
    validation = Subset(dataset, indices[:n_validation])
    training = Subset(dataset, indices[n_validation:])
    return (
        DataLoader(training, batch_size=batch_size, shuffle=True),
        DataLoader(validation, batch_size=batch_size, shuffle=False),
    )


def evaluate_loss(model, data_loader):
    """Average the classifier loss over one data loader."""
    model.eval()
    total = 0.0
    n_events = 0
    with torch.no_grad():
        for x, nu, labels in data_loader:
            x = x.to(device)
            nu = nu.to(device)
            labels = labels.to(device)
            loss = equation17_loss(model(x, nu), labels)
            total += loss.item() * x.shape[0]
            n_events += x.shape[0]
    return total / n_events


def clip_parameters(model, limit):
    """Apply the loose clipping used by the reference implementation."""
    with torch.no_grad():
        for parameter in model.parameters():
            parameter.clamp_(-limit, limit)


def train_model(model, sample, split_seed):
    """Train one Taylor ratio model and restore its best checkpoint."""
    training_loader, validation_loader = make_data_loaders(sample, split_seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"training": [], "validation": []}
    best_state = copy.deepcopy(model.state_dict())
    best_validation = np.inf
    stale_epochs = 0

    for epoch in range(max_epochs):
        model.train()
        training_total = 0.0
        n_training = 0
        for x, nu, labels in training_loader:
            x = x.to(device)
            nu = nu.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = equation17_loss(model(x, nu), labels)
            loss.backward()
            optimizer.step()
            clip_parameters(model, weight_clip)
            training_total += loss.item() * x.shape[0]
            n_training += x.shape[0]

        training_loss = training_total / n_training
        validation_loss = evaluate_loss(model, validation_loader)
        history["training"].append(training_loss)
        history["validation"].append(validation_loss)

        if validation_loss < best_validation - 1e-7:
            best_validation = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 0 or (epoch + 1) % 50 == 0:
            print(
                f"epoch {epoch + 1:4d}: train={training_loss:.6f}, "
                f"validation={validation_loss:.6f}"
            )
        if stale_epochs >= early_stopping_patience:
            print(f"early stopping at epoch {epoch + 1}")
            break

    model.load_state_dict(best_state)
    return model, history

## 7. Train the Linear and Quadratic Models

These are two independent optimizations. The quadratic model learns its two coefficient networks jointly, but it cannot alter the coefficient learned by the linear-only model.

In [8]:
torch.manual_seed(seed)
linear_model = TaylorRatioNetwork(order=1, width=hidden_units)
start = time.time()
linear_model, linear_history = train_model(
    linear_model, linear_sample, split_seed=seed
)
print(f"linear training time: {time.time() - start:.1f} s")

torch.manual_seed(seed + 1)
quadratic_model = TaylorRatioNetwork(order=2, width=hidden_units)
start = time.time()
quadratic_model, quadratic_history = train_model(
    quadratic_model, quadratic_sample, split_seed=seed + 1
)
print(f"quadratic training time: {time.time() - start:.1f} s")

epoch    1: train=24.997488, validation=24.991221
epoch   50: train=24.966638, validation=24.955109


KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, history, title in zip(
    axes,
    [linear_history, quadratic_history],
    ["linear model", "quadratic model"],
):
    ax.plot(history["training"], label="training")
    ax.plot(history["validation"], label="validation")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.set_ylabel("Eq. 17 loss")
    ax.legend()
fig.tight_layout();

## 8. Inspect the Coefficient Functions

The first panel directly compares the independently trained $\widehat\delta_1^L$ and $\widehat\delta_1^Q$. This makes any deformation of the quadratic model's linear coefficient explicit rather than hiding it in the reconstructed ratio.

In [ ]:
def predict_coefficients(model, x):
    """Evaluate a model's coefficient networks."""
    model.eval()
    x_tensor = torch.from_numpy(
        np.ascontiguousarray(x, dtype=np.float32)
    ).to(device)
    with torch.no_grad():
        delta1, delta2 = model.coefficients(x_tensor)
    delta1 = delta1.cpu().numpy().reshape(-1)
    if delta2 is not None:
        delta2 = delta2.cpu().numpy().reshape(-1)
    return delta1, delta2


def predict_log_ratio(model, x, nu):
    """Evaluate a fitted neural nuisance-ratio model."""
    model.eval()
    x_tensor = torch.from_numpy(
        np.ascontiguousarray(x, dtype=np.float32)
    ).to(device)
    nu_tensor = torch.full_like(x_tensor, float(nu))
    with torch.no_grad():
        prediction = model(x_tensor, nu_tensor)
    return prediction.cpu().numpy().reshape(-1)


x_flat = x_grid.reshape(-1)
delta1_linear, _ = predict_coefficients(linear_model, x_grid)
delta1_quadratic, delta2_quadratic = predict_coefficients(
    quadratic_model, x_grid
)
delta1_true = exact_delta1(x_flat)
delta2_true = exact_delta2(x_flat)

In [ ]:
print(
    "linear delta1 RMSE:",
    np.sqrt(np.mean((delta1_linear - delta1_true)**2)),
)
print(
    "quadratic delta1 RMSE:",
    np.sqrt(np.mean((delta1_quadratic - delta1_true)**2)),
)
print(
    "quadratic delta2 RMSE:",
    np.sqrt(np.mean((delta2_quadratic - delta2_true)**2)),
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_flat, delta1_true, color="black", lw=2, label="analytic")
axes[0].plot(
    x_flat, delta1_linear, "o", fillstyle="none", markevery=12,
    label="linear model delta1",
)
axes[0].plot(
    x_flat, delta1_quadratic, "s", fillstyle="none", markevery=12,
    label="quadratic model delta1",
)
axes[0].set_title("linear coefficient")
axes[0].set_xlabel("x")
axes[0].set_ylabel("delta1(x)")
axes[0].legend()

axes[1].plot(x_flat, delta2_true, color="black", lw=2, label="analytic")
axes[1].plot(
    x_flat, delta2_quadratic, "o", fillstyle="none", markevery=12,
    label="quadratic model delta2",
)
axes[1].set_title("quadratic coefficient")
axes[1].set_xlabel("x")
axes[1].set_ylabel("delta2(x)")
axes[1].legend()
fig.tight_layout();

## 9. Reconstruct the Pointwise Log-Ratio

The linear prediction always comes from the independent linear model. The quadratic prediction is the direct output of the separately trained two-network model.

In [ ]:
test_nu_values = [-0.6, -0.3, -0.1, 0.1, 0.3, 0.6]
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True)

for ax, nu in zip(axes.reshape(-1), test_nu_values):
    exact = exact_log_ratio(x_flat, nu)
    linear = predict_log_ratio(linear_model, x_grid, nu)
    quadratic = predict_log_ratio(quadratic_model, x_grid, nu)
    ax.plot(x_flat, exact, "--", lw=2, label="exact")
    ax.plot(x_flat, linear, lw=1.5, label="linear NN")
    ax.plot(x_flat, quadratic, lw=1.5, label="quadratic NN")
    ax.set_title(f"nu={nu:+.1f}")
    ax.set_xlabel("x")
    ax.set_ylabel("log r(x; nu)")

axes[0, 0].legend()
fig.tight_layout();

## 10. Fig. 3 Analogue

Evaluate both models on a fresh central reference sample and average the event weights inside logarithmic bins. Dashed curves are the exact pointwise ratio, filled squares are the quartic bin-level monitor, and empty circles are the binned neural-network reconstruction.

In [ ]:
def exact_bin_log_ratio(bin_edges, nu):
    """Calculate log N_bin(nu) / N_bin(0) analytically."""
    low = np.asarray(bin_edges[:-1])
    high = np.asarray(bin_edges[1:])
    scale = np.exp(float(nu))
    varied_probability = np.exp(-low / scale) - np.exp(-high / scale)
    central_probability = np.exp(-low) - np.exp(-high)
    return np.log(varied_probability / central_probability)


def binned_learned_log_ratio(x_reference, log_ratio, bin_edges):
    """Average learned event weights inside each reference bin."""
    x_reference = np.asarray(x_reference).reshape(-1)
    event_weights = np.exp(np.asarray(log_ratio).reshape(-1))
    values = []
    for low, high in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (x_reference >= low) & (x_reference < high)
        values.append(
            np.log(np.mean(event_weights[in_bin])) if np.any(in_bin) else np.nan
        )
    return np.asarray(values)


def fit_quartic_bin_monitor(bin_edges, nuisance_scan):
    """Fit the analytic bin response as a quartic polynomial in nu."""
    bin_values = np.vstack([
        exact_bin_log_ratio(bin_edges, nu) for nu in nuisance_scan
    ])
    return [
        np.polyfit(nuisance_scan, bin_values[:, index], 4)
        for index in range(bin_values.shape[1])
    ]

In [ ]:
rng_figure = np.random.default_rng(seed + 2)
x_reference = sample_exponential(20_000, 0.0, rng_figure)
figure3_nu_values = [-0.6, -0.3, -0.1, 0.1, 0.3, 0.6]
figure3_nu_scan = np.linspace(-0.6, 0.6, 49)
figure3_x_min = 1e-2
figure3_bin_edges = np.geomspace(figure3_x_min, x_max, 19)
figure3_bin_centers = np.sqrt(
    figure3_bin_edges[:-1] * figure3_bin_edges[1:]
)
figure3_bin_xerr = np.vstack([
    figure3_bin_centers - figure3_bin_edges[:-1],
    figure3_bin_edges[1:] - figure3_bin_centers,
])
figure3_x_dense = np.geomspace(figure3_x_min, x_max, 250)
quartic_coefficients = fit_quartic_bin_monitor(
    figure3_bin_edges, figure3_nu_scan
)
colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(figure3_nu_values)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for index, (nu, color) in enumerate(zip(figure3_nu_values, colors)):
    exact = exact_log_ratio(figure3_x_dense, nu)
    linear_events = predict_log_ratio(linear_model, x_reference, nu)
    quadratic_events = predict_log_ratio(quadratic_model, x_reference, nu)
    linear_binned = binned_learned_log_ratio(
        x_reference, linear_events, figure3_bin_edges
    )
    quadratic_binned = binned_learned_log_ratio(
        x_reference, quadratic_events, figure3_bin_edges
    )
    quartic_binned = np.asarray([
        np.polyval(coefficient, nu) for coefficient in quartic_coefficients
    ])

    for ax, learned_binned in zip(axes, [linear_binned, quadratic_binned]):
        ax.plot(
            figure3_x_dense, exact, "--", color=color, lw=1.8,
            label="exact" if index == 0 else "_nolegend_",
        )
        ax.errorbar(
            figure3_bin_centers, quartic_binned, xerr=figure3_bin_xerr,
            fmt="s", color=color, ms=3.5, elinewidth=0.9, capsize=0,
            linestyle="none", alpha=0.85,
            label="quartic binned" if index == 0 else "_nolegend_",
        )
        ax.plot(
            figure3_bin_centers, learned_binned, "o", color=color,
            markerfacecolor="white", fillstyle="none", ms=5, mew=1.2,
            label="NN binned" if index == 0 else "_nolegend_",
        )

axes[0].set_title("independent linear model")
axes[1].set_title("two-network quadratic model")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlim(figure3_x_min, x_max)
    ax.set_ylim(-2.2, 1.8)
    ax.set_xlabel("x")
    ax.set_ylabel("log r(x; nu)")
    ax.axhline(0.0, color="0.75", lw=1)

for nu, color in zip(figure3_nu_values, colors):
    axes[1].plot([], [], color=color, lw=2, label=f"nu={nu:+.1f}")
axes[1].legend(fontsize=8, ncol=2, loc="lower left")
fig.suptitle("Fig. 3 analogue: PyTorch nuisance networks")
fig.tight_layout();